# 🌬️ AQF - Exploration Initiale et Nettoyage (ETL)

> *« La donnée brute est le minerai, le nettoyage est le raffinage. »*

Ce notebook initialise notre pipeline d'analyse pour la qualité de l'air en France (Focus Île-de-France 2023).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration esthétique
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Chargement des Données
Nous chargeons les fichiers NO2 et PM10 de 2023.

In [ ]:
df_no2 = pd.read_csv('../data/raw/airparif_2023_no2.csv')
df_pm10 = pd.read_csv('../data/raw/airparif_2023_pm10.csv')

print(f"NO2 : {df_no2.shape[0]} lignes chargées.")
print(f"PM10 : {df_pm10.shape[0]} lignes chargées.")

df_no2.head()

## 2. Nettoyage et Transformation (Wrangling)
Un bon analyste s'assure que les dates sont bien interprétées.

In [ ]:
# Conversion des dates
df_no2['date'] = pd.to_datetime(df_no2['date'])
df_pm10['date'] = pd.to_datetime(df_pm10['date'])

# Extraction de features temporelles (Le génie est dans les détails)
df_no2['mois'] = df_no2['date'].dt.month
df_no2['jour_semaine'] = df_no2['date'].dt.day_name()
df_no2['heure'] = df_no2['date'].dt.hour

df_no2.head()

## 3. Analyse Descriptive : Les Pics de Pollution
Quelle est la station la plus polluée en moyenne ?

In [ ]:
moyenne_station = df_no2.groupby('station')['valeur'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=moyenne_station.index, y=moyenne_station.values, hue=moyenne_station.index, palette='Reds_r', legend=False)
plt.title('Concentration Moyenne de NO2 par Station (2023)')
plt.ylabel('NO2 (µg/m³)')
plt.axhline(40, color='red', linestyle='--', label='Seuil OMS Annuel')
plt.legend()
plt.show()

## 4. Corrélation Temporelle : Le Cycle Circadien
Voyons l'impact du trafic (pics matin et soir).

In [ ]:
cycle_horaire = df_no2.groupby('heure')['valeur'].mean()

plt.figure(figsize=(10, 5))
sns.lineplot(x=cycle_horaire.index, y=cycle_horaire.values, marker='o', color='blue')
plt.title('Cycle Horaire Moyen de la Pollution au NO2')
plt.xlabel('Heure de la journée')
plt.ylabel('Concentration (µg/m³)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()